# Inspect Power BI TMDL Model and Source Workbook
This notebook loads the existing TMDL table definitions and examines the external Excel workbook used by the model.

In [ ]:
import zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

workspace_root = Path(r'c:/Users/Ahmed Elbadrawy/Downloads/datax_update/update_datax')
model_dir = workspace_root / 'no1.SemanticModel' / 'definition'
workbook_path = Path(r'D:/work/projects/data x projects/New folder/ROW_DATA.xlsx')
output_path = workspace_root / 'workbook_headers.txt'

keywords = {
    'SalesDetailsKey', 'SalesHeaderKey', 'SalesOrderNumber', 'ProductKey', 'ProductSubcategoryKey',
    'OrderQuantity', 'UnitPrice', 'ExtendedAmount', 'CustomerKey', 'GeographyKey', 'CountryCode',
    'RegionKey', 'Year', 'MonthNo', 'UnitCost', 'SubcategoryName', 'CategoryName',
    'CityName', 'StateCode', 'StateName', 'CountryName', 'Region', 'Continent'
}
lines = []
lines.append(f'Model directory: {model_dir}')
lines.append(f'Workbook path: {workbook_path}')
lines.append('Tables in model:')
for path in sorted((model_dir / 'tables').glob('*.tmdl')):
    lines.append(f' - {path.name}')

with zipfile.ZipFile(workbook_path) as z:
    workbook_xml = ET.fromstring(z.read('xl/workbook.xml'))
    ns = {'x': 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    sheets = []
    for element in workbook_xml.findall('.//x:sheets/x:sheet', ns):
        rid = element.attrib.get('{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id')
        sheets.append((element.attrib['name'], rid))
    rels = ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))
    rel_map = {r.attrib['Id']: r.attrib['Target'] for r in rels.findall('{http://schemas.openxmlformats.org/package/2006/relationships}Relationship')}
    shared_strings = []
    if 'xl/sharedStrings.xml' in z.namelist():
        shared_strings = [t.text or '' for si in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}t') for t in [si]]
    for name, rid in sheets:
        target = rel_map.get(rid)
        if not target or not target.startswith('worksheets/'):
            continue
        sheet_xml = ET.fromstring(z.read('xl/' + target))
        header = None
        for row in sheet_xml.findall('.//{http://schemas.openxmlformats.org/spreadsheetml/2006/main}row'):
            cells = []
            for cell in row.findall('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}c'):
                v = cell.find('{http://schemas.openxmlformats.org/spreadsheetml/2006/main}v')
                if v is None:
                    value = ''
                else:
                    value = v.text or ''
                    if cell.attrib.get('t') == 's':
                        value = shared_strings[int(value)]
                cells.append(value)
            if any(cells):
                header = cells
                break
        if header is None:
            continue
        hits = [h for h in header if h in keywords]
        if hits:
            lines.append(f'Sheet: {name} target: {target} hits: {len(hits)}')
            lines.append(' header: ' + ','.join(header))
            lines.append(' matched: ' + ','.join(hits))

output_path.write_text('\n'.join(lines), encoding='utf-8')
print('Wrote', output_path)


Model directory: c:\Users\Ahmed Elbadrawy\Downloads\datax_update\update_datax\no1.SemanticModel\definition
Workbook path: D:\work\projects\data x projects\New folder\ROW_DATA.xlsx
Tables in model:
 - Customer.tmdl
 - DateTableTemplate_3388ec4d-112f-4723-a0f5-a3aa34f6e1af.tmdl
 - Geography.tmdl
 - LocalDateTable_48c1b41e-6b15-4848-bd06-707a7e72c3f4.tmdl
 - LocalDateTable_4d6012b8-7bf2-4228-90d9-1d1629c1b93f.tmdl
 - LocalDateTable_5a1ff29e-47e4-45a0-b193-de94024a258f.tmdl
 - LocalDateTable_7c050121-23c8-41c5-9675-f26a5d33c2f4.tmdl
 - LocalDateTable_9cf27755-9b11-45fe-8654-0dde946f1928.tmdl
 - Product.tmdl
 - ProductCostHistory.tmdl
 - ProductSubcategory.tmdl
 - Region.tmdl
 - SalesDetails.tmdl
 - SalesHeader.tmdl
 - SalesReturns.tmdl
Sheet: Sheet1 target: worksheets/sheet1.xml
 header count: 7 hits: 2
 header fields: ['CustomerKey', 'GeographyKey', 'BusinessType', 'Customer', 'NumberEmployees', 'AnnualRevenue', 'YearOpened']
Sheet: Sheet2 target: worksheets/sheet2.xml
 header count: 5 hi